In [ ]:
!pip install fastprop

In [ ]:
# @title train Model
!fastprop train \
  --input-file train.csv \
  --smiles-column SMILES \
  --target-columns LogS \
  --output-directory fastprop_results \
  --problem-type regression \
  --number-epochs 100 \
  --batch-size 256

In [ ]:
# @title run predictions
import os
import pandas as pd

# A. Create a temporary SMILES file from your benchmark file
# We assume 'Tight_set.csv' is your SC2 benchmark file
test_df = pd.read_csv("sc2.csv")
test_df["SMILES"].to_csv("test_smiles.txt", index=False, header=False)

# B. Find the trained model folder (Fastprop adds a timestamp)
base_dir = "fastprop_results"
# Find the latest directory created inside fastprop_results
latest_run = max([os.path.join(base_dir, d) for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))], key=os.path.getmtime)
checkpoint_dir = os.path.join(latest_run, "checkpoints")

print(f"Using model from: {checkpoint_dir}")

# C. Run Prediction using the bash command
# We use $checkpoint_dir to pass the python variable to the shell
!fastprop predict \
  -sf test_smiles.txt \
  -ds all \
  -o fastprop_predictions.csv \
  $checkpoint_dir

In [ ]:
# @title check score
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

gt_df = pd.read_csv("sc2.csv")
pred_df = pd.read_csv("fastprop_predictions.csv")

gt_df['fastprop_pred'] = pred_df['task_0'].values

target_col = "Solubility" if "Solubility" in gt_df.columns else "LogS"
y_true = gt_df[target_col]
y_pred = gt_df['fastprop_pred']

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print(f"RMSE: {rmse:.4f}")
print(f"R2: {r2:.4f}")
